# Spark Execution Fundamentals (Understanding the Engine - AQE Disabled)

This interactive notebook lets you run PySpark operations step-by-step to observe how the **Spark Engine** processes jobs under the hood. 

In this run, we have **disabled Adaptive Query Execution (AQE)** so you can observe the raw, static physical execution plans and shuffle stages compiled at submission time. We have also set the log level to **ERROR** to keep the console printouts clean.

As you execute each cell, you can explore the live **Spark UI** at **[http://localhost:4040](http://localhost:4040)**.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize Spark Session
# Set shuffle partitions to 4 to make the stages easily readable in Spark UI
# Disable Adaptive Query Execution (AQE) to observe the static execution model
spark = SparkSession.builder \
    .appName("SparkEngineDemo-Notebook") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Active! AQE Disabled, Log Level set to ERROR.")
print("👉 Spark UI exposes at: http://localhost:4040")

## Step 1: Read Orders Dataset (No Schema Inference)

First, we read the `sample_orders.csv` dataset. We configure `.option("inferSchema", "false")` and `.option("header", "true")`.

### 🔍 Spark UI Observation:
1. Run the cell below.
2. Check the **Jobs** tab in the Spark UI. You will notice that **exactly 1 job (Job 0)** has been created.
3. **Why?** Even though Spark is lazy and type inference is disabled, because we specified `.option("header", "true")`, Spark must eagerly submit 1 job to read the first line of the CSV file to resolve the header column names.

In [ ]:
orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("sample_orders.csv")

orders_df.printSchema()
print("\nOrders loaded. Check the Spark UI - exactly 1 job (Job 0) should be created!")

## Step 2: Read Customers Dataset (With Schema Inference)

Next, we read the `sample_customers.csv` dataset in a separate cell, this time setting `.option("inferSchema", "true")` and `.option("header", "true")`.

### 🔍 Spark UI Observation:
1. Run the cell below.
2. Go to the **Jobs** tab in the Spark UI. You will notice that **exactly 2 jobs (Job 1 and Job 2)** have been created.
3. **Why?** Because both `header` and `inferSchema` are set to `true`, Spark must perform two eager read operations: one job to retrieve the column headers, and a second job to scan the rows and infer their data types.

In [ ]:
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("sample_customers.csv")

customers_df.printSchema()
print("\nCustomers loaded. Check the Spark UI - exactly 2 jobs (Job 1 and Job 2) should be created!")

## Step 3: Define Narrow Transformations & Observe Lazy Evaluation

We define a filter and select transformation on our orders dataframe. These are **Narrow Transformations** because data does not move across executors.

### 🔍 Spark UI Observation:
1. Run the cell below.
2. Refresh the Spark UI. No new jobs appear because Spark records the DAG path lazily without initiating computation.

In [ ]:
orders_filtered = orders_df \
    .filter(F.col("Amount") > 150.0) \
    .select("OrderID", "CustomerID", "Amount")

print("Transformation defined! Check Spark UI—no new jobs should appear.")

## Step 4: Trigger Action 1 (Narrow Transformations)

We trigger the `.count()` action on the filtered orders.

### 🔍 Spark UI Observation:
1. Run the cell below.
2. A new **Job 3** will appear in the Spark UI.
3. Click on **Job 3** to view its details.
4. **Observe the Stages**: It contains exactly **1 Stage**.
5. **Observe the DAG**: You will see a straight-line flow containing `FileScan` -> `Filter` -> `Project` (Select) -> `Exchange/HashAggregate` all in one block. 
6. **Why?** Since no data shuffle was required, Spark executed the entire pipeline inside a single Stage.

In [ ]:
count = orders_filtered.count()
print(f"Filtered Orders Count = {count}")
print("\nCheck Job 3 in Spark UI. Observe that this job has exactly 1 Stage and a straight-line DAG.")

## Step 5: Wide Transformation (GroupBy / Shuffling - AQE Disabled)

We group the orders by `CustomerID` and sum the `Amount`. `groupBy` is a **Wide Transformation**.

💡 **Why we write instead of show()**:
If we run `orders_grouped.show()`, Spark's optimizer inserts a `Limit` operator in the execution path since `.show()` only asks for the first 20 rows. This shortcutting optimization alters the plan and hides the real stages. To see the true physical execution plan, shuffles, and partition flow, we **write the full output** to a no-operation sink (`noop`) to consume all records.

### 🔍 Spark UI Observation:
1. Run the cell below.
2. Look at **Job 4** in the Spark UI.
3. **Observe the Stages**: Notice that Job 4 has **2 Stages**.
4. **Observe the Shuffle metrics**:
   * Stage A shows **Shuffle Write**.
   * Stage B shows **Shuffle Read**.
5. **Inspect the DAG**: Click on Job 4. Notice a blue box boundary called `Exchange` separating the two stages. 
   * **AQE Off Behavior**: Unlike the Adaptive Query Execution model which wraps stages dynamically and optimizes partitions, here the plan compiles to exactly 2 static stages. The Exchange boundary is static, and the DAG shows the direct static query plan.

In [ ]:
orders_grouped = orders_df \
    .groupBy("CustomerID") \
    .agg(F.round(F.sum("Amount"), 2).alias("TotalSales"))

# Trigger Action via complete write to no-operation format
orders_grouped.write.mode("overwrite").format("noop").save()
print("Check Job 4 in Spark UI. Notice it is split into 2 Stages at the Static Shuffle Exchange Boundary.")

## Step 6: Complex DAG (Join Transformation - Broadcast Hash Join)

We join the aggregated orders with the customer dataset on `CustomerID`.

### 🔍 Spark UI Observation:
1. Run the cell below.
2. Go to the **Jobs** tab in the Spark UI. You will notice that **exactly 2 new jobs (Job 5 and Job 6)** have been spawned.
3. **Why did one write action trigger two jobs?**
   * **Job 5 (2 Stages)**: This has the description `$anonfun$withThreadLocalCaptured$1...`. This is a helper job created by Spark to collect the `customers` dataset and build the relation hashtable on the driver for a **BroadcastHashJoin** (since the customers dataset is extremely small, under the 10MB broadcast threshold).
   * **Job 6 (1 Stage)**: This is the main write action (`save at...`). It receives the broadcasted table on the executors, executes the map-side join, and writes to `noop` in a single stage with **no physical shuffles**!
4. Click on **Job 6** and look at the DAG. You will see the broadcast exchange side merging directly into the query stream.

In [ ]:
final_report = orders_grouped.join(customers_df, on="CustomerID", how="inner") \
    .select("CustomerID", "CustomerName", "TotalSales", "Country")

# Trigger Action via complete write to no-operation format
final_report.write.mode("overwrite").format("noop").save()
print("Check Jobs 5 & 6 in the Spark UI. Observe the helper broadcast job and the 1-stage join save job!")

## Step 7: Exploring the SQL / DataFrame Tab in Spark UI

While the **Jobs** tab shows the execution timeline, the **SQL/DataFrame** tab exposes the underlying optimizer plans and execution details. Every time you trigger an action on a DataFrame (like `.show()`, `.write()`, or `.count()`), Spark logs the operation details under the **SQL/DataFrame** tab.

### 🔍 Spark UI Observation:
1. Run the cell below to register temporary SQL views and run a Spark SQL query.
2. Open the **SQL/DataFrame** tab in your browser (`http://localhost:4040/SQL/`).
3. Click on the link for the query you just executed (labeled `save at <ipython-input-...>`).
4. **Inspect the Query Plan Visualization**:
   * **FileScan csv**: Click on the scan boxes to see the path, input schema, partition stats, and metrics (like `number of output rows`).
   * **Filter**: Observe the filter node condition (e.g., `CAST(Amount AS DOUBLE) > 100.0`).
   * **Exchange**: Observe the shuffle key, output partitions (4), and bytes transferred.
   * **SortMergeJoin** / **BroadcastHashJoin**: Observe the join key and type (Inner Join).
5. Click on the **details** toggle at the bottom of the page to inspect Spark's four logical and physical optimizer phases:
   * **Parsed Logical Plan**: The raw syntax representation of the query.
   * **Analyzed Logical Plan**: Schema-resolved plan (verifies relations and types).
   * **Optimized Logical Plan**: Spark CatalystOptimizer optimizations (like filter pushdowns and projection pruning).
   * **Physical Plan**: The actual execution instructions sent to the cluster executors.

In [ ]:
# Register temporary views for SQL operations
orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")

# Execute a SQL Join Aggregation query
sql_df = spark.sql("""
    SELECT 
        c.Country,
        ROUND(SUM(CAST(o.Amount AS DOUBLE)), 2) as TotalRevenue
    FROM orders o
    JOIN customers c ON o.CustomerID = c.CustomerID
    WHERE CAST(o.Amount AS DOUBLE) > 100.0
    GROUP BY c.Country
    ORDER BY TotalRevenue DESC
""")

# Trigger Action via complete write to no-operation format
sql_df.write.mode("overwrite").format("noop").save()
print("SQL/DataFrame query executed. Explore its visualization and physical details in the Spark UI!")

## Step 8: Shutdown Spark Session

In [ ]:
spark.stop()
print("Spark Session stopped successfully.")